# Compare posterior samples from two differently-formatted HDF5 files

This notebook overlays a corner plot comparing posterior samples from two HDF5 files that use *different* internal layouts, e.g.:

- A **DINGO importance-sampling result**: a single compound/structured dataset (commonly called `"samples"`) with columns like `chirp_mass, mass_ratio, chi_1, chi_2, ..., weights, log_prob, log_prior`.
- A **GWTC-style PE result** (bilby / PESummary release file): a nested group such as `"C00:IMRPhenomXO4a/posterior_samples"` containing a compound dataset with the full precessing parameter set (`mass_1, mass_2, a_1, a_2, tilt_1, tilt_2, phi_12, phi_jl, ...`).

### Important caveat

These two files were generated with **different spin parameterizations**: the DINGO file only has aligned-spin components (`chi_1`, `chi_2`), while the GWTC file has full precessing spin angles (`a_1, a_2, tilt_1, tilt_2, ...`). There is therefore no exact 1-to-1 map for every precessing spin angle.

This notebook instead builds the largest *comparable* set of detector-frame parameters by:
1. using directly-shared columns (`chirp_mass, mass_ratio, ra, dec, psi, phase, theta_jn, geocent_time, luminosity_distance`), and
2. deriving `mass_1` / `mass_2` (from `chirp_mass` & `mass_ratio`) and `chi_eff` (mass-weighted aligned spin) on both sides so precession-only angles don't block the comparison.

Adjust `PARAMS_WANTED` further down if you want a different / smaller / larger set, or if your two files happen to share the full 15-parameter precessing set, in which case just list all 15 there directly.

**Also note:** `geocent_time` is *not* directly comparable between these two example files — in the DINGO file it's a small offset (~±0.02 s) relative to the trigger time, while in the GWTC file it's an absolute GPS time (~1.3877×10⁹). Either drop it from `PARAMS_WANTED` or shift the DINGO column by the trigger GPS time before comparing.

**Requires:** `h5py`, `numpy`, `pandas`, `corner`, `matplotlib`
```
pip install h5py numpy pandas corner matplotlib
```

In [ ]:
import h5py
import numpy as np
import pandas as pd
import corner
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

## 1. Generic HDF5 posterior loader (handles BOTH layouts automatically)

In [ ]:
def _search(h5obj, required_keys):
    """Recursively search an open h5py File/Group for the first dataset or
    group of 1-D datasets that contains all `required_keys`."""
    for name, obj in h5obj.items():
        if isinstance(obj, h5py.Dataset):
            fields = obj.dtype.names
            if fields and all(k in fields for k in required_keys):
                return "dataset", obj
        elif isinstance(obj, h5py.Group):
            keys = set(obj.keys())
            if required_keys.issubset(keys):
                return "group", obj
            found = _search(obj, required_keys)
            if found is not None:
                return found
    return None


def load_posterior(path, group_path=None,
                    required_keys=("chirp_mass", "mass_ratio")):
    """
    Load a posterior-sample table from an HDF5 file into a pandas
    DataFrame, regardless of whether it's stored as:
      - a single compound/structured dataset (DINGO "samples" style), or
      - a group of individual 1-D datasets (bilby/GWTC "posterior_samples"
        style).

    Parameters
    ----------
    path : str
        Path to the .hdf5 / .h5 file.
    group_path : str, optional
        Explicit path inside the file, e.g.
        "C00:IMRPhenomXO4a/posterior_samples". If omitted, the file is
        searched automatically for a table containing `required_keys`.
    required_keys : tuple of str
        Columns used to identify the correct table during auto-search.
    """
    required = set(required_keys)
    with h5py.File(path, "r") as f:
        if group_path is not None:
            obj = f[group_path]
            kind = "dataset" if isinstance(obj, h5py.Dataset) else "group"
            target = obj
        else:
            found = _search(f, required)
            if found is None:
                raise ValueError(
                    f"Could not auto-locate a posterior table containing "
                    f"{required_keys} in \'{path}\'. Pass group_path=... "
                    f"explicitly (inspect the file with h5py/HDFView first)."
                )
            kind, target = found

        if kind == "dataset":
            data = target[()]
            df = pd.DataFrame({n: data[n] for n in data.dtype.names})
        else:
            df = pd.DataFrame({
                k: target[k][()] for k in target.keys()
                if isinstance(target[k], h5py.Dataset) and target[k].ndim == 1
            })
    return df


def get_weights(df):
    """Return normalized importance-sampling weights if present,
    otherwise None (equal-weight posterior)."""
    if "weights" in df.columns:
        w = df["weights"].to_numpy(dtype=float)
        return w / w.sum()
    return None

## 2. Derive quantities needed to make the two parameterizations comparable

In [ ]:
def add_derived_quantities(df):
    """Add mass_1, mass_2 (from chirp_mass & mass_ratio) and chi_eff
    (mass-weighted aligned spin) if not already present, using whichever
    spin columns are available."""
    df = df.copy()

    if "mass_1" not in df.columns and {"chirp_mass", "mass_ratio"} <= set(df.columns):
        mc, q = df["chirp_mass"].to_numpy(), df["mass_ratio"].to_numpy()
        total_mass = mc * (1.0 + q) ** 1.2 / q ** 0.6
        df["mass_1"] = total_mass / (1.0 + q)
        df["mass_2"] = total_mass * q / (1.0 + q)

    if "chi_eff" not in df.columns:
        m1 = df.get("mass_1")
        m2 = df.get("mass_2")
        if m1 is not None and m2 is not None:
            if {"chi_1", "chi_2"} <= set(df.columns):          # aligned-spin file
                s1z, s2z = df["chi_1"].to_numpy(), df["chi_2"].to_numpy()
            elif {"spin_1z", "spin_2z"} <= set(df.columns):    # precessing file, cartesian
                s1z, s2z = df["spin_1z"].to_numpy(), df["spin_2z"].to_numpy()
            elif {"a_1", "cos_tilt_1", "a_2", "cos_tilt_2"} <= set(df.columns):
                s1z = df["a_1"].to_numpy() * df["cos_tilt_1"].to_numpy()
                s2z = df["a_2"].to_numpy() * df["cos_tilt_2"].to_numpy()
            else:
                s1z = s2z = None
            if s1z is not None:
                df["chi_eff"] = (m1.to_numpy() * s1z + m2.to_numpy() * s2z) / (m1 + m2).to_numpy()
    return df

## 3. Overlay corner plot

In [ ]:
def make_comparison_corner(df1, w1, label1, df2, w2, label2, params,
                            outfile="corner_comparison.png"):
    missing1 = [p for p in params if p not in df1.columns]
    missing2 = [p for p in params if p not in df2.columns]
    if missing1 or missing2:
        raise ValueError(
            f"Requested params missing -> file1: {missing1}, file2: {missing2}"
        )

    data1 = df1[params].to_numpy()
    data2 = df2[params].to_numpy()

    # Shared axis ranges so both posteriors are shown on the same scale
    ranges = []
    for i, p in enumerate(params):
        lo = min(np.nanmin(data1[:, i]), np.nanmin(data2[:, i]))
        hi = max(np.nanmax(data1[:, i]), np.nanmax(data2[:, i]))
        pad = 0.05 * (hi - lo if hi > lo else 1.0)
        ranges.append((lo - pad, hi + pad))

    fig = corner.corner(
        data1, weights=w1, labels=params, range=ranges,
        color="C0", plot_datapoints=False, plot_density=False,
        fill_contours=True, levels=(0.5, 0.9), hist_kwargs={"density": True},
    )
    corner.corner(
        data2, weights=w2, fig=fig, range=ranges,
        color="C1", plot_datapoints=False, plot_density=False,
        fill_contours=True, levels=(0.5, 0.9), hist_kwargs={"density": True},
    )

    fig.legend(
        handles=[
            mlines.Line2D([], [], color="C0", label=label1),
            mlines.Line2D([], [], color="C1", label=label2),
        ],
        loc="upper right", fontsize=16, frameon=False,
        bbox_to_anchor=(0.98, 0.98),
    )

    fig.savefig(outfile, dpi=150, bbox_inches="tight")
    print(f"Saved: {outfile}")
    return fig

## 4. Load the two files\n\nEdit the paths (and `GWTC_GROUP`, if needed) below.

In [ ]:
# --- edit these paths -------------------------------------------------
FILE_DINGO = "GW231226_101520_data0_1387620938-3_importance_sampling.hdf5"
FILE_GWTC = "IGWN-GWTC4p1-18965dda8_5-GW231226_101520-combined_PEDataRelease.h5"
GWTC_GROUP = "C00:IMRPhenomXO4a/posterior_samples"  # set None to auto-search
# -----------------------------------------------------------------------

df_a = load_posterior(FILE_DINGO)                       # auto-detects "samples"
df_b = load_posterior(FILE_GWTC, group_path=GWTC_GROUP)

w_a = get_weights(df_a)   # DINGO importance-sampling weights
w_b = get_weights(df_b)   # None -> equal-weight GWTC posterior

df_a = add_derived_quantities(df_a)
df_b = add_derived_quantities(df_b)

df_a.head()

In [ ]:
df_b.head()

## 5. Pick the shared parameter set and plot

In [ ]:
# Preferred detector-frame parameter list. Anything not present in BOTH
# files after the derivation step above is dropped automatically, with
# a printed warning -- if your two files really do share the full
# precessing 15-parameter set, just list all 15 here directly.
PARAMS_WANTED = [
    "chirp_mass", "mass_ratio", "mass_1", "mass_2",
    "chi_eff", "theta_jn", "phase",
    "ra", "dec", "psi", "geocent_time", "luminosity_distance",
]

params = [p for p in PARAMS_WANTED if p in df_a.columns and p in df_b.columns]
dropped = [p for p in PARAMS_WANTED if p not in params]
if dropped:
    print(f"Note: dropping params not present in both files: {dropped}")
print(f"Plotting {len(params)} shared detector-frame parameters: {params}")

In [ ]:
fig = make_comparison_corner(
    df_a, w_a, "DINGO (IS)",
    df_b, w_b, "GWTC PE",
    params,
    outfile="corner_comparison.png",
)
fig